
# N06: High-Confidence Pseudo-Labeling
**Objective:** Push the 0.94986 public baseline limit without external datasets by augmenting the training matrix with extremely high-confidence test set predictions (>99% probability).


In [ ]:

import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings('ignore')

ID_COL, TARGET_COL = "id", "health_condition"
CLASSES = ("at-risk", "fit", "unhealthy")
SEED = 42
N_FOLDS = 5
PSEUDO_CONFIDENCE_THRESHOLD = 0.99


In [ ]:

# 1. Load Data
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e7/test.csv')

print(f"Initial Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Feature Engineering
for df in [train_df, test_df]:
    df['sleep_bin'] = pd.qcut(df['sleep_duration'], q=5, duplicates='drop').astype(str)
    df['stress_sleep_interact'] = df['stress_level'].astype(str) + '_' + df['sleep_bin']

cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality', 'stress_sleep_interact']
num_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']

le_target = LabelEncoder()
y_train = le_target.fit_transform(train_df[TARGET_COL])

X_train_raw = train_df[cat_cols + num_cols].copy()
X_test_raw = test_df[cat_cols + num_cols].copy()

for col in cat_cols:
    X_train_raw[col] = X_train_raw[col].fillna('Missing').astype(str)
    X_test_raw[col] = X_test_raw[col].fillna('Missing').astype(str)

num_imputer = SimpleImputer(strategy='median')
X_train_num_raw = num_imputer.fit_transform(train_df[num_cols])
X_test_num_raw = num_imputer.transform(test_df[num_cols])


In [ ]:

# 2. Step A: Generate Base Test Set Probabilities using N04 CatBoost Logic
tst_probs = np.zeros((len(test_df), 3))
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print("--- Step A: Generating Baseline Test Set Posterior Probabilities ---")

for fold, (tr_i, va_i) in enumerate(skf.split(X_train_raw, y_train)):
    X_tr_cat, X_va_cat = X_train_raw[cat_cols].iloc[tr_i], X_train_raw[cat_cols].iloc[va_i]
    X_tr_num, X_va_num = X_train_num_raw[tr_i], X_train_num_raw[va_i]
    y_tr, y_va = y_train[tr_i], y_train[va_i]
    
    fold_te_tr, fold_te_val, fold_te_test = [], [], []
    
    for col in cat_cols:
        crosstab = pd.crosstab(X_tr_cat[col], y_tr, normalize='index')
        mapping = crosstab.to_dict('index')
        mean_mapping = crosstab.mean().to_dict()
        
        tr_mapped = X_tr_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_tr.append(pd.DataFrame(tr_mapped.tolist()).values)
        
        te_mapped = X_test_raw[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_test.append(pd.DataFrame(te_mapped.tolist()).values)
        
    X_tr_full = np.hstack([X_tr_num, np.hstack(fold_te_tr)])
    X_te_full = np.hstack([X_test_num_raw, np.hstack(fold_te_test)])
    
    class_counts = np.bincount(y_tr)
    weights = [len(y_tr) / (len(class_counts) * c) for c in class_counts]
    
    m = CatBoostClassifier(
        iterations=1000, learning_rate=0.03, depth=6,
        class_weights=weights, random_seed=SEED+fold, verbose=0,
        task_type="CPU"
    )
    
    m.fit(X_tr_full, y_tr)
    tst_probs += m.predict_proba(X_te_full) / N_FOLDS
    print(f"Completed Base Fold {fold + 1}/{N_FOLDS}")


In [ ]:

# 3. Step B: Extract Extremely Confident Pseudo-Labels
max_probs = tst_probs.max(axis=1)
predicted_classes = tst_probs.argmax(axis=1)

pseudo_mask = max_probs >= PSEUDO_CONFIDENCE_THRESHOLD
num_pseudo = pseudo_mask.sum()

print(f"\n--- Step B: Pseudo-Label Extraction ---")
print(f"Threshold: {PSEUDO_CONFIDENCE_THRESHOLD}")
print(f"Extracted Pseudo-Labeled Rows: {num_pseudo} ({num_pseudo/len(test_df)*100:.2f}% of test set)")

pseudo_X_cat = X_test_raw[pseudo_mask]
pseudo_X_num = X_test_num_raw[pseudo_mask]
pseudo_y = predicted_classes[pseudo_mask]

# Create Augmented Dataset
aug_X_cat_df = pd.concat([X_train_raw[cat_cols], pseudo_X_cat[cat_cols]], ignore_index=True)
aug_X_num_mat = np.vstack([X_train_num_raw, pseudo_X_num])
aug_y = np.concatenate([y_train, pseudo_y])

print(f"\nAugmented Training Shape: {aug_X_num_mat.shape[0]} rows")


In [ ]:

# 4. Step C: Train Final Architecture on Augmented Matrix
final_tst_probs = np.zeros((len(test_df), 3))
oof_preds = np.zeros(len(aug_y))

skf_aug = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
print("\n--- Step C: Augmented Training ---")

for fold, (tr_i, va_i) in enumerate(skf_aug.split(aug_X_cat_df, aug_y)):
    X_tr_cat, X_va_cat = aug_X_cat_df.iloc[tr_i], aug_X_cat_df.iloc[va_i]
    X_tr_num, X_va_num = aug_X_num_mat[tr_i], aug_X_num_mat[va_i]
    y_tr, y_va = aug_y[tr_i], aug_y[va_i]
    
    fold_te_tr, fold_te_val, fold_te_test = [], [], []
    
    for col in cat_cols:
        crosstab = pd.crosstab(X_tr_cat[col], y_tr, normalize='index')
        mapping = crosstab.to_dict('index')
        mean_mapping = crosstab.mean().to_dict()
        
        tr_mapped = X_tr_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_tr.append(pd.DataFrame(tr_mapped.tolist()).values)
        
        va_mapped = X_va_cat[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_val.append(pd.DataFrame(va_mapped.tolist()).values)
        
        te_mapped = X_test_raw[col].map(mapping).apply(lambda x: x if isinstance(x, dict) else mean_mapping)
        fold_te_test.append(pd.DataFrame(te_mapped.tolist()).values)
        
    X_tr_full = np.hstack([X_tr_num, np.hstack(fold_te_tr)])
    X_va_full = np.hstack([X_va_num, np.hstack(fold_te_val)])
    X_te_full = np.hstack([X_test_num_raw, np.hstack(fold_te_test)])
    
    class_counts = np.bincount(y_tr)
    weights = [len(y_tr) / (len(class_counts) * c) for c in class_counts]
    
    m = CatBoostClassifier(
        iterations=1500, learning_rate=0.03, depth=6,
        class_weights=weights, random_seed=SEED+fold, verbose=0,
        task_type="CPU"
    )
    
    m.fit(X_tr_full, y_tr, eval_set=(X_va_full, y_va), early_stopping_rounds=100)
    
    val_preds = m.predict(X_va_full).flatten()
    oof_preds[va_i] = val_preds
    final_tst_probs += m.predict_proba(X_te_full) / N_FOLDS
    
    print(f"  Fold {fold + 1}/{N_FOLDS} Augmented Balanced Accuracy: {balanced_accuracy_score(y_va, val_preds):.5f}")

print(f"\nFinal OOF Augmented Balanced Accuracy: {balanced_accuracy_score(aug_y, oof_preds):.5f}")


In [ ]:

# 5. Output Final Predictions
sub_df = pd.DataFrame({
    ID_COL: test_df[ID_COL].astype("int64"),
    TARGET_COL: [CLASSES[i] for i in final_tst_probs.argmax(axis=1)]
})

sub_df.to_csv("submission.csv", index=False)
print("Exported Pseudo-Labeled submission.csv")
